In [ ]:
import pandas as pd
import os
os.environ['USE_PYGEOS'] = '0'
import geopandas as gpd
import networkx as nx
from pathlib import Path
import networkx as nx
import matplotlib.pyplot as plt
import numpy as np
pd.set_option('display.max_columns', 500)

from sklearn.preprocessing import MinMaxScaler
import sys
import pyspace
import libpysal as lps
from scipy.spatial import cKDTree
from libpysal.weights.distance import get_points_array
from esda import fdr
from importlib import reload
pd.set_option('display.max_rows', 500)
reload(pyspace)
import seaborn as sns
from esda.moran import Moran
# sns.set_theme(font = 'Helvetica')
%matplotlib inline
from numba import NumbaDeprecationWarning
import warnings
import os
# Suppress NumbaDeprecationWarning
warnings.filterwarnings("ignore", category=NumbaDeprecationWarning)
from utils import read_data, plot_covariate_distributions, plot_match, compare_balance, sizeof_fmt, optimize_memory_df, plot_categorical_proportional_diff, compute_mean_differences_and_proportions, love_plot, sensitivity_analysis_k_neighbors

In [ ]:
data_folder  = Path('../../SanteIntegra/Data/')
results_folder = Path('../results/')
output_folder = Path('../output')
model_folder = results_folder/'Models'

if not os.path.exists(model_folder):
    os.makedirs(model_folder)

In [ ]:
df_open = pd.read_parquet(data_folder/'processed/df_treated_open.parquet.gzip')
df_5years = pd.read_parquet(data_folder/'processed/df_treated_5years.parquet.gzip')

In [ ]:
df_open.groupby(['NOANNEE','CDPHYSSEXE'])['PRESTATIONS_BRUTES_AOS'].agg({"mean","std"}).unstack(level=1).div(1000).round(1)

In [ ]:
test = pd.concat([df_open.groupby(['NOANNEE','CDPHYSSEXE'])['PRESTATIONS_BRUTES_AOS'].agg({"mean","std"}).unstack().div(1000).round(1),
df_open.groupby(['NOANNEE','age_group'],observed=True)['PRESTATIONS_BRUTES_AOS'].agg({"mean","std"}).unstack().div(1000).round(1),
df_open.groupby(['NOANNEE','ssep3_q'],observed=True)['PRESTATIONS_BRUTES_AOS'].agg({"mean","std"}).unstack().div(1000).round(1),
df_open.groupby(['NOANNEE','Language'],observed=True)['PRESTATIONS_BRUTES_AOS'].agg({"mean","std"}).unstack().div(1000).round(1),
df_open.groupby(['NOANNEE','CAREMODEL'],observed=True)['PRESTATIONS_BRUTES_AOS'].agg({"mean","std"}).unstack().div(1000).round(1),
df_open.groupby(['NOANNEE','MTFRANCHISECOUV'],observed=True)['PRESTATIONS_BRUTES_AOS'].agg({"mean","std"}).unstack().div(1000).round(1)], axis=1).T.reset_index()
test.columns = ['Statistic', 'Categories', '2017', '2018', '2019', '2020', '2021']

# Pivot the DataFrame to make each row represent one year and one sex
df_pivot = test.pivot(index='Categories', columns='Statistic')

for year in ['2017', '2018', '2019', '2020', '2021']:
    df_pivot[year] = df_pivot[(year, 'mean')].astype(str) + " (" + df_pivot[(year, 'std')].astype(str) + ")"
df_pivot.columns = df_pivot.columns.droplevel(1)  # Drops the upper level, leaving only the years
df_final = df_pivot.reset_index()
df_final = df_final.loc[:,~df_final.columns.duplicated()].copy()
df_final.to_clipboard()

In [ ]:
def percentile_25(x):
    return np.percentile(x, 25)

def percentile_75(x):
    return np.percentile(x, 75)

def median(x):
    return np.median(x)

In [ ]:
test = pd.concat([df_open.groupby(['NOANNEE','CDPHYSSEXE'])['PRESTATIONS_BRUTES_LCA'].agg({"mean","std"}).unstack().round(1),
df_open.groupby(['NOANNEE','age_group'],observed=True)['PRESTATIONS_BRUTES_LCA'].agg({"mean","std"}).unstack().round(1),
df_open.groupby(['NOANNEE','ssep3_q'],observed=True)['PRESTATIONS_BRUTES_LCA'].agg({"mean","std"}).unstack().round(1),
df_open.groupby(['NOANNEE','Language'],observed=True)['PRESTATIONS_BRUTES_LCA'].agg({"mean","std"}).unstack().round(1),
df_open.groupby(['NOANNEE','CAREMODEL'],observed=True)['PRESTATIONS_BRUTES_LCA'].agg({"mean","std"}).unstack().round(1),
df_open.groupby(['NOANNEE','MTFRANCHISECOUV'],observed=True)['PRESTATIONS_BRUTES_LCA'].agg({"mean","std"}).unstack().round(1)], axis=1).T.reset_index()
test.columns = ['Statistic', 'Categories', '2017', '2018', '2019', '2020', '2021']

# Pivot the DataFrame to make each row represent one year and one sex
df_pivot = test.pivot(index='Categories', columns='Statistic')

for year in ['2017', '2018', '2019', '2020', '2021']:
    df_pivot[year] = df_pivot[(year, 'mean')].astype(str) + " (" + df_pivot[(year, 'std')].astype(str) + ")"
df_pivot.columns = df_pivot.columns.droplevel(1)  # Drops the upper level, leaving only the years
df_final = df_pivot.reset_index()
df_final = df_final.loc[:,~df_final.columns.duplicated()].copy()
df_final.to_clipboard()

In [ ]:
pd.concat([df_open.groupby(['NOANNEE','CDPHYSSEXE'])['PRESTATIONS_BRUTES_LCA'].agg({"median","std"}).unstack().round(1),
df_open.groupby(['NOANNEE','age_group'],observed=True)['PRESTATIONS_BRUTES_LCA'].agg({"median","std"}).unstack().round(1),
df_open.groupby(['NOANNEE','ssep3_q'],observed=True)['PRESTATIONS_BRUTES_LCA'].agg({"median","std"}).unstack().round(1),
df_open.groupby(['NOANNEE','CDLANGUE'],observed=True)['PRESTATIONS_BRUTES_LCA'].agg({"median","std"}).unstack().round(1),
df_open.groupby(['NOANNEE','CAREMODEL'],observed=True)['PRESTATIONS_BRUTES_LCA'].agg({"median","std"}).unstack().round(1)], axis=1).T.reset_index()

In [ ]:
df_uuid = df_open[['uuid']]
df_uuid = df_uuid.drop_duplicates().reset_index(drop=True)

### Figure S2A - Trends in healthcare expenditures (2017-2021)

In [ ]:
yearly_data_AOS = df_5years.groupby('NOANNEE')['PRESTATIONS_BRUTES_AOS'].sum().div(1000000).reset_index()
yearly_data_LCA = df_5years.groupby('NOANNEE')['PRESTATIONS_BRUTES_LCA'].sum().div(1000000).reset_index()
yearly_data_CAM = df_5years.groupby('NOANNEE')['PRESTATIONS_BRUTES_CAM'].sum().div(1000000).reset_index()

# Create a long format DataFrame for easier plotting with seaborn
combined_data = pd.concat([
    yearly_data_AOS.rename(columns={'PRESTATIONS_BRUTES_AOS': 'Values'}).assign(Type='CM (MHI)'),
    yearly_data_CAM.rename(columns={'PRESTATIONS_BRUTES_CAM': 'Values'}).assign(Type='CAM (MHI)'),
    yearly_data_LCA.rename(columns={'PRESTATIONS_BRUTES_LCA': 'Values'}).assign(Type='CAM (SI)')
])

# Adjusted for FacetGrid with custom x-ticks
def plot_facet_grid(data, x, y, facet_col, x_label, y_label):
    g = sns.FacetGrid(data, row=facet_col, sharey=False, height=4, aspect=2)
    g.map_dataframe(sns.lineplot, x=x, y=y, marker='o')
    g.set_axis_labels(x_label, y_label)

    # Set x-ticks for each facet
    for ax in g.axes.flatten():
        ax.set_xticks(data[x].unique())  # Assuming x-values are year numbers
        ax.set_xticklabels(data[x].unique(), rotation=45)
        ax.grid(True)

# Combine and rename data as per previous example for combined_data
plot_facet_grid(
    combined_data,
    'NOANNEE', 'Values', 'Type',
    'Year', 'Total annual expenditures (millions CHF)'
)
plt.savefig(results_folder/'Fig S2A - Archive.png', dpi = 300, bbox_inches='tight')

In [ ]:
# Instead of summing, keep individual observations for boxplots
# Create long format DataFrame with individual expenditures
aos_data = df_open[['NOANNEE', 'PRESTATIONS_BRUTES_AOS']].copy()
aos_data = aos_data[aos_data['PRESTATIONS_BRUTES_AOS'] > 0]  # Only non-zero for meaningful boxplots
aos_data['Values'] = aos_data['PRESTATIONS_BRUTES_AOS'] / 1000  # Convert to thousands CHF
aos_data['Insurance'] = 'CM (MHI)'

cam_mhi_data = df_open[['NOANNEE', 'PRESTATIONS_BRUTES_CAM']].copy()
cam_mhi_data = cam_mhi_data[cam_mhi_data['PRESTATIONS_BRUTES_CAM'] > 0]
cam_mhi_data['Values'] = cam_mhi_data['PRESTATIONS_BRUTES_CAM'] / 1000
cam_mhi_data['Insurance'] = 'CAM (MHI)'

lca_data = df_open[['NOANNEE', 'PRESTATIONS_BRUTES_LCA']].copy()
lca_data = lca_data[lca_data['PRESTATIONS_BRUTES_LCA'] > 0]
lca_data['Values'] = lca_data['PRESTATIONS_BRUTES_LCA'] / 1000
lca_data['Insurance'] = 'CAM (SI)'

# Combine all data
combined_data = pd.concat([
    aos_data[['NOANNEE', 'Values', 'Insurance']],
    cam_mhi_data[['NOANNEE', 'Values', 'Insurance']],
    lca_data[['NOANNEE', 'Values', 'Insurance']]
])

# Create boxplot function
def plot_boxplot_facet(data, x, y, facet_col, x_label, y_label):
    g = sns.FacetGrid(data, row=facet_col, sharey=False, height=4, aspect=2)
    g.map_dataframe(sns.boxplot, x=x, y=y, showfliers=False)  # Hide outliers
    g.set_axis_labels(x_label, y_label)
    
    for ax in g.axes.flatten():
        ax.set_xticklabels(ax.get_xticklabels(), rotation=45)
        ax.grid(True, alpha=0.3)
    
    return g

# Create the plot
plot_boxplot_facet(
    combined_data,
    'NOANNEE', 'Values', 'Insurance',
    'Year', 'Annual expenditures (thousands CHF)'
)

plt.savefig(output_folder/'Fig S2A - Evolution_costs_boxplot.png', dpi=300, bbox_inches='tight')

### Figure S2B - Trends in healthcare expenditures (2017-2021)

In [ ]:
yearly_data_AOS = df_5years.groupby('NOANNEE')['PRESTATIONS_BRUTES_AOS'].sum().div(1000000).reset_index()
yearly_data_LCA = df_5years.groupby('NOANNEE')['PRESTATIONS_BRUTES_LCA'].sum().div(1000000).reset_index()
yearly_data_CAM = df_5years.groupby('NOANNEE')['PRESTATIONS_BRUTES_CAM'].sum().div(1000000).reset_index()

# Normalize data

yearly_data_AOS['PRESTATIONS_BRUTES_AOS'] = yearly_data_AOS['PRESTATIONS_BRUTES_AOS'] / yearly_data_AOS['PRESTATIONS_BRUTES_AOS'].iloc[0] * 100
yearly_data_LCA['PRESTATIONS_BRUTES_LCA'] = yearly_data_LCA['PRESTATIONS_BRUTES_LCA'] / yearly_data_LCA['PRESTATIONS_BRUTES_LCA'].iloc[0] * 100
yearly_data_CAM['PRESTATIONS_BRUTES_CAM'] = yearly_data_CAM['PRESTATIONS_BRUTES_CAM'] / yearly_data_CAM['PRESTATIONS_BRUTES_CAM'].iloc[0] * 100

# Plot normalized data
fig, ax = plt.subplots(figsize=(8, 4))
sns.lineplot(data=yearly_data_AOS, x='NOANNEE', y='PRESTATIONS_BRUTES_AOS', marker='o', label='CM (MHI)', ax=ax)
sns.lineplot(data=yearly_data_LCA, x='NOANNEE', y='PRESTATIONS_BRUTES_LCA', marker='o', label='CAM (SI)', ax=ax)
sns.lineplot(data=yearly_data_CAM, x='NOANNEE', y='PRESTATIONS_BRUTES_CAM', marker='o', label='CAM (MHI)', ax=ax)
ax.set_xticks(yearly_data_CAM['NOANNEE'])  # Assuming x-values are year numbers
ax.set_xticklabels(yearly_data_CAM['NOANNEE'].astype('string'), rotation=45)
ax.grid(True)

plt.xlabel('Year')
plt.ylabel('Evolution of annual expenditures (%)')  # Update label to indicate these are percentages
plt.grid(True)
# plt.legend(title='')
# plt.show()
plt.savefig(results_folder/'Fig S2B - Evolution_costs_perc_baseline.png', dpi = 300, bbox_inches='tight')

In [ ]:
# Calculate per-capita expenditures first
yearly_data_AOS = df_open.groupby('NOANNEE').agg({
    'PRESTATIONS_BRUTES_AOS': ['sum', 'count']
}).reset_index()
yearly_data_AOS.columns = ['NOANNEE', 'total_AOS', 'count_AOS']
yearly_data_AOS['avg_AOS'] = yearly_data_AOS['total_AOS'] / yearly_data_AOS['count_AOS'] / 1000  # Per person in thousands

yearly_data_LCA = df_open.groupby('NOANNEE').agg({
    'PRESTATIONS_BRUTES_LCA': ['sum', 'count']
}).reset_index()
yearly_data_LCA.columns = ['NOANNEE', 'total_LCA', 'count_LCA']
yearly_data_LCA['avg_LCA'] = yearly_data_LCA['total_LCA'] / yearly_data_LCA['count_LCA'] / 1000

yearly_data_CAM = df_open.groupby('NOANNEE').agg({
    'PRESTATIONS_BRUTES_CAM': ['sum', 'count']
}).reset_index()
yearly_data_CAM.columns = ['NOANNEE', 'total_CAM', 'count_CAM']
yearly_data_CAM['avg_CAM'] = yearly_data_CAM['total_CAM'] / yearly_data_CAM['count_CAM'] / 1000

# Normalize per-capita data to 2017 baseline
yearly_data_AOS['avg_AOS_norm'] = yearly_data_AOS['avg_AOS'] / yearly_data_AOS['avg_AOS'].iloc[0] * 100
yearly_data_LCA['avg_LCA_norm'] = yearly_data_LCA['avg_LCA'] / yearly_data_LCA['avg_LCA'].iloc[0] * 100
yearly_data_CAM['avg_CAM_norm'] = yearly_data_CAM['avg_CAM'] / yearly_data_CAM['avg_CAM'].iloc[0] * 100

# Plot normalized per-capita data
fig, ax = plt.subplots(figsize=(8, 6))
sns.lineplot(data=yearly_data_AOS, x='NOANNEE', y='avg_AOS_norm', marker='o', label='CM (MHI)', ax=ax)
sns.lineplot(data=yearly_data_LCA, x='NOANNEE', y='avg_LCA_norm', marker='o', label='CAM (SI)', ax=ax)
sns.lineplot(data=yearly_data_CAM, x='NOANNEE', y='avg_CAM_norm', marker='o', label='CAM (MHI)', ax=ax)

ax.set_xticks(yearly_data_CAM['NOANNEE'])
ax.set_xticklabels(yearly_data_CAM['NOANNEE'].astype('string'), rotation=45)
ax.grid(True)
plt.xlabel('Year')
plt.ylabel('Evolution of average annual expenditures per person (%)')  # Updated label
plt.legend(title='Prestation type')
plt.savefig(results_folder/'Fig S2B.png', dpi=300, bbox_inches='tight')

### Cost distribution after transformation

#### AOS

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(16, 4))

g = sns.kdeplot(data=df_open, x="PRESTATIONS_BRUTES_AOS", ax = axs[0])
g.set_xlabel('Yearly CM claims amount (MHI) (CHF)')
g = sns.kdeplot(data=df_open, x="ihs_cost_aos", ax = axs[1])
g.set_xlabel('IHS transformed yearly CM claims amount (MHI) (CHF)')
g = sns.kdeplot(data=df_open[df_open.PRESTATIONS_BRUTES_AOS > df_open.MTFRANCHISECOUV], x="ihs_cost_aos", ax = axs[2])
g.set_xlabel('IHS transformed yearly CM claims amount (MHI) (CHF)')
plt.savefig(output_folder/'IHS_transformation_procedure_AOS.png', dpi = 300, bbox_inches='tight')

#### LCA

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(16, 4))

g = sns.kdeplot(data=df_open, x="PRESTATIONS_BRUTES_LCA", ax = axs[0])
g.set_xlabel('Yearly CAM claims amount (SI) (CHF)')
g = sns.kdeplot(data=df_open, x="ihs_cost_lca", ax = axs[1])
g.set_xlabel('IHS transformed yearly CAM claims amount (SI) (CHF)')
g = sns.kdeplot(data=df_open[df_open.PRESTATIONS_BRUTES_LCA > df_open.MTFRANCHISECOUV], x="ihs_cost_lca", ax = axs[2])
g.set_xlabel('IHS transformed yearly CAM claims amount (SI) (CHF)')
plt.savefig(output_folder/'IHS_transformation_procedure_LCA.png', dpi = 300, bbox_inches='tight')

## Analyses de sous-groupes

- Répondre à des questions spécifiques
- Augmenter l'homogénéité de la population étudiée

In [ ]:
df_healthy = pd.read_parquet(data_folder/'processed/df_healthy_nominors_open.parquet.gzip')
df_multimorbid = pd.read_parquet(data_folder/'processed/df_multimorbidity_nominors_open.parquet.gzip')
df_cancer = pd.read_parquet(data_folder/'processed/df_cancer_nominors_open.parquet.gzip')

In [ ]:
df_healthy_5years = pd.read_parquet(data_folder/'processed/df_healthy_nominors_5years.parquet.gzip')
df_multimorbid_5years = pd.read_parquet(data_folder/'processed/df_multimorbidity_nominors_5years.parquet.gzip')
df_cancer_5years = pd.read_parquet(data_folder/'processed/df_cancer_nominors_5years.parquet.gzip')

## Total spending by category

In [ ]:
variable_names = pd.DataFrame({"old": ['multimorbidity','ssep3_q','Urbanicity_simple','n_atc','n_flags','NBAGE',"NBAGE_std",'age_group', "ssep3_std",'ssep3', 'region_DE', 'region_FR', 'region_IT','urb_Urbain','urb_Périurbain','Asthma_PCG', 'Cancer_PCG', 'Diabetes_PCG', 'Epilepsy_PCG',
       'Glaucoma_PCG', 'HIV_AIDS_PCG', 'Heart_disease_PCG',
       'Hypertension_related_PCG', 'Immune_PCG', 'Inflammatory_PCG',
       'Mental_PCG', 'Other_PCG', 'Pain_PCG', 'Parkinson_PCG', 'Thyroid_PCG', "SEX_F",'SEX','CDPHYSSEXE','LANG', "cds_std",'cds','LANG_FR','D_MEDIC_B','D_MEDIC_S','D_MEDIC_B_std','D_MEDIC_S_std','DEDUCTIBLE_above_500','E_std','N_std','E_std:N_std','PRESTATIONS_BRUTES_ATC','PRESTATIONS_TOTAL','PRESTATIONS_BRUTES_AOS','PRESTATIONS_BRUTES_LCA','PRESTATIONS_BRUTES_CAM','PRESTATIONS_BRUTES_AMBULATOIRE','PRESTATIONS_BRUTES_STATIONNAIRE','PRESTATIONS_ACCIDENT','PRESTATIONS_DISEASE','PRESTATIONS_BIRTH','MTFRANCHISECOUV','mean_pm10','mean_no2','mean_pm25','mean_ndvi','mean_lst','mean_carnight'],
                           "new": ['Multimorbidity','Swiss-SEP','Urbanicity','Number of ATC','Number of PCG flags','Age',"Age", 'Age Group', "SES index",'SES index','German', 'French', 'Italian','Urban','Periurban', 'Asthma', 'Cancer', 'Diabetes', 'Epilepsy', 'Glaucoma', 'HIV/AIDS',
       'Heart disease', 'Hypertension related', 'Immune', 'Inflammatory',
       'Mental', 'Other', 'Pain', 'Parkinson', 'Thyroid', "Sex (Female)",'Sex','Sex','Langage', "CDS",'CDS','French speaker','Access to prim. care med.','Access to spec. med.','Access to prim. care med.','Access to spec. med.','Deductible (>500)','E','N','E:N','Drug-related claims (CHF)','Total claims amount (CHF)','CM claims (MHI) (CHF)','CAM claims (SI) (CHF)','CAM claims (MHI) (CHF)','Ambulatory claims (CHF)','Stationary claims (CHF)','Accident-related claims (CHF)','Disease-related claims (CHF)','Birth-related claims (CHF)','Deductible','PM10','NO2','PM25','NDVI','LST','Nighttime car noise']})
rename_dict = variable_names.set_index('old')['new'].to_dict()

def update_variable_names(summary_table, variable_names, table_type):
    name_mapper = variable_names.set_index('old')['new'].to_dict()
    if table_type == 'summary':
        name_mapper = {f"{key}, mean (SD)": f"{value}, mean (SD)" for key, value in name_mapper.items()}
    elif table_type == 'categorical':
        name_mapper = {f"{key}, n (%)": f"{value}, n (%)" for key, value in name_mapper.items()}
    summary_table = summary_table.rename(index=name_mapper)
    return summary_table

In [ ]:
df_open_long = df_open.set_index('NOANNEE')[['PRESTATIONS_BRUTES_AOS','PRESTATIONS_BRUTES_CAM','PRESTATIONS_BRUTES_LCA']].stack().reset_index()

df_open_long.columns = ['NOANNEE','cat_spending','amount']

# sns.barplot(df_treated_filtered_long, x="NOANNEE", y="amount", hue = 'cat_spending')

In [ ]:
# Assuming df_treated_filtered_long is your DataFrame after manipulation
df_open_long['cat_spending'] = df_open_long['cat_spending'].replace({
    'PRESTATIONS_BRUTES_AOS': 'Conventional medicine (MHI)',
    'PRESTATIONS_BRUTES_CAM': 'Complementary medicine (MHI)',
    'PRESTATIONS_BRUTES_LCA': 'Complementary medicine (SI)'
})

In [ ]:
palette = {
    'Conventional medicine (MHI)': '#4E79A7',
    'Complementary medicine (MHI)': '#59A14F',
    'Complementary medicine (SI)': '#F28E2B'
}

In [ ]:
# Now plot with Seaborn
g = sns.barplot(data=df_open_long, x="NOANNEE", y="amount", hue='cat_spending', palette=palette)
# plt.ylim([0,10000])
# g.set_yscale("log")
# the non-logarithmic labels you want
# ticks = [1, 10, 100, 1000, 10000, 100000]
# g.set_yticks(ticks)
# g.set_yticklabels(ticks)
g.set_xlabel('Year')
plt.legend(title='')
# If you want to further customize the legend, you can use plt.legend:
# plt.legend(title='S')
plt.ylabel('Average yearly claims amount (CHF)')
# To ensure the plot displays properly in Jupyter notebooks
plt.savefig(output_folder/'Avg_expenditures.png', dpi = 300, bbox_inches='tight')

In [ ]:
# Now plot with Seaborn
g = sns.barplot(data=df_open_long, x="NOANNEE", y="amount", hue='cat_spending', palette=palette)
# plt.ylim([0,10000])
g.set_yscale("log")
# the non-logarithmic labels you want
ticks = [1, 10, 100, 1000, 10000, 100000]
g.set_yticks(ticks)
g.set_yticklabels(ticks)
g.set_xlabel('Year')
plt.legend(title='')
# If you want to further customize the legend, you can use plt.legend:
# plt.legend(title='S')
plt.ylabel('Average yearly claims amount (CHF)')
# To ensure the plot displays properly in Jupyter notebooks
plt.savefig(output_folder/'Avg_expenditures_log.png', dpi = 300, bbox_inches='tight')

In [ ]:
# First subplot
g = sns.lineplot(x='NOANNEE', y='amount', hue='cat_spending', data=df_open_long, marker='o')
g.set_yscale("log")
# the non-logarithmic labels you want
ticks = [1, 10, 100, 1000, 10000, 100000]
g.set_yticks(ticks)
g.set_yticklabels(ticks)
g.set_xlabel('Year')
plt.legend(title='')

In [ ]:
df_open[df_open.PRESTATIONS_BRUTES_CAM > 0].groupby('NOANNEE').uuid.nunique().plot.bar()

In [ ]:
df_open[df_open.PRESTATIONS_BRUTES_LCA > 0].groupby('NOANNEE').uuid.nunique().plot.bar()

In [ ]:
g = sns.barplot(df_open[df_open.PRESTATIONS_BRUTES_CAM > 0], x="NOANNEE", y="PRESTATIONS_BRUTES_CAM", color = '#59A14F')
g.set_ylabel('Average yearly claims amount (CHF)')
# g.legend(title='Complementary medicine (LAMal) for users')
g.set_xlabel('Year')
g.set_ylim([0,1500])
plt.savefig(output_folder/'Avg_expenditures_CAM_users.png', dpi = 300, bbox_inches='tight')

In [ ]:
df_open[df_open.PRESTATIONS_BRUTES_CAM > 0].groupby('year')['PRESTATIONS_BRUTES_CAM'].describe()

In [ ]:
sns.barplot(df_open[df_open.PRESTATIONS_BRUTES_LCA > 0], x="NOANNEE", y="PRESTATIONS_BRUTES_LCA")

### Number of individuals by cumulated years of treatment

In [ ]:
ttmt_month_gp_uuid = df_open.groupby('uuid')['n_month_lca_by_patient'].sum().to_dict()
df_open['cumulated_n_month_lca_by_patient'] = df_open['uuid'].map(ttmt_month_gp_uuid)
df_uuid['cumulated_n_month_lca_by_patient'] = df_uuid['uuid'].map(ttmt_month_gp_uuid)

In [ ]:
df_open.groupby('uuid')['PRESTATIONS_BRUTES_LCA'].sum().plot.hist(bins = 100)

In [ ]:
# df_treated_filtered.groupby('treatment_cumulated_lca_cam').uuid.nunique().plot.bar()
from matplotlib.ticker import FuncFormatter

# First, perform your groupby operation and reset the index to turn it into a DataFrame
df_aggregated = df_open.groupby('treatment_cumulated_lca_cam')['uuid'].nunique().reset_index()

# Rename the columns for clarity if desired
df_aggregated.columns = ['Treatment', 'UniqueCount']
# Now, use Seaborn to create a bar plot
def thousands_separator(x, pos):
    return '{:,.0f}'.format(x)  # The '0f' ensures no decimal places

# Assuming df_aggregated is your DataFrame after the groupby and reset_index
sns.barplot(x='Treatment', y='UniqueCount', data=df_aggregated, palette = 'Blues')

# Get the current axis
ax = plt.gca()

# Set the formatter for the y-axis to use your custom formatter
ax.yaxis.set_major_formatter(FuncFormatter(thousands_separator))

# Optional: add labels and title for clarity
plt.xlabel('Number of years in the treatment group (CAM claims amount > 0) \n (MHI and SI combined)')
plt.ylabel('Number of individuals')
# plt.title('Number of individuals by Treatment')
plt.savefig(output_folder/'Number_users_CAM_cumulated_years_open.png', dpi = 300, bbox_inches='tight')

In [ ]:
# First, perform your groupby operation and reset the index to turn it into a DataFrame
df_aggregated = df_5years.groupby('treatment_cumulated_lca_cam')['uuid'].nunique().reset_index()

# Rename the columns for clarity if desired
df_aggregated.columns = ['Treatment', 'UniqueCount']
# Now, use Seaborn to create a bar plot
def thousands_separator(x, pos):
    return '{:,.0f}'.format(x)  # The '0f' ensures no decimal places

# Assuming df_aggregated is your DataFrame after the groupby and reset_index
sns.barplot(x='Treatment', y='UniqueCount', data=df_aggregated, palette = 'Blues')

# Get the current axis
ax = plt.gca()

# Set the formatter for the y-axis to use your custom formatter
ax.yaxis.set_major_formatter(FuncFormatter(thousands_separator))

# Optional: add labels and title for clarity
plt.xlabel('Number of years in the treatment group (CAM claims amount > 0) \n (MHI and SI combined)')
plt.ylabel('Number of individuals')
# plt.title('Number of individuals by Treatment')
plt.savefig(output_folder/'Number_users_CAM_cumulated_years_5years.png', dpi = 300, bbox_inches='tight')

In [ ]:
df_aggregated['perc_count'] = (df_aggregated['UniqueCount']/df_aggregated['UniqueCount'].sum())*100

In [ ]:
df_open.groupby('cumulated_n_month_lca_by_patient').uuid.nunique().plot.bar(figsize = (12,8))

In [ ]:
df_uuid['cumulated_n_month_lca_by_patient'].median()

## What generates spending in AOS?

#### Time-varying covariates (proxies for health status)
- Age
- Sex
- Deductible
- Education, income (Swiss SES index)
- Access to healthcare
    - Captured directly through measured access (existing variables : d_medic_s, d_medic)
    - Captured indirectly through urban contexts (urban, periurban, rural)
- Disease state : CDS, PCGs, HOSPITALISATIONS

#### Time-invariant covariates
- Sociocultural norms:
    - Captured through the canton of residence and spoken langage
- Policies :
    - Captured through the canton of residence 
- Environmental factors:
    - Air quality (NO2, PM10, PM2.5)
    - Vegetation (NDVI)
    - Noise (dB)

#### Time-varying treatment
- Treatment

### Health status -> cost, treatment

In [ ]:
# Create a 2x2 grid of subplots
fig, axs = plt.subplots(2, 2, figsize=(9, 6))

# First subplot
sns.boxplot(data=df_open, y="PRESTATIONS_BRUTES_AOS", x='n_flags', hue='n_flags', 
            width=.6, palette="vlag", showfliers=False, legend=False, ax=axs[0, 0])

# Second subplot
sns.boxplot(data=df_open, y="PRESTATIONS_DISEASE", x='n_flags', hue='n_flags', 
            width=.6, palette="vlag", showfliers=False, legend=False, ax=axs[0, 1])

# Third subplot
sns.boxplot(data=df_open, y="PRESTATIONS_BRUTES_LCA", x='n_flags', hue='n_flags', 
            width=.6, palette="vlag", showfliers=False, legend=False, ax=axs[1, 0])

# Fourth subplot
sns.boxplot(data=df_open, y="PRESTATIONS_BRUTES_AOS", x='locdrhosp', hue='locdrhosp', 
            width=.6, palette="vlag", showfliers=False, legend=False, ax=axs[1, 1])

# Adjust layout
plt.tight_layout()
plt.show()

In [ ]:
# Create a 2x2 grid of subplots
fig, axs = plt.subplots(2, 2, figsize=(6, 4))

# First subplot
sns.boxplot(data=df_open, y="PRESTATIONS_BRUTES_AOS", x='multimorbidity', hue='multimorbidity', width=.6, palette="vlag", showfliers=False, ax=axs[0,0])

# Second subplot
sns.boxplot(data=df_open, y="PRESTATIONS_BRUTES_LCA", x='multimorbidity', hue='multimorbidity', width=.6, palette="vlag", showfliers=False, ax=axs[0,1])

# Third subplot
sns.barplot(data=df_open, y="PRESTATIONS_BRUTES_CAM", x='multimorbidity', hue='multimorbidity', width=.6, palette="vlag", ax=axs[1,0])

# Fourth subplot
sns.boxplot(data=df_open, y="PRESTATIONS_DISEASE", x='multimorbidity', hue='multimorbidity', width=.6, palette="vlag", showfliers=False, ax=axs[1,1])


# Adjust layout
plt.tight_layout()
plt.show()

In [ ]:
df_5years.groupby('multimorbidity')['PRESTATIONS_BRUTES_CAM'].mean()

In [ ]:
df_open.groupby('multimorbidity')['PRESTATIONS_BRUTES_CAM'].mean()

#### Remarques

- Les personnes avec des comorbidités ne sont pas celles qui utilisent plus de LCA
- Les personnes avec des comorbidités dépensent bcp plus en AOS (logique)

In [ ]:
# Create a 2x2 grid of subplots
fig, axs = plt.subplots(2, 2, figsize=(10, 6))

# First subplot
g = sns.boxplot(data=df_open, y="PRESTATIONS_BRUTES_AOS", x='CDPHYSSEXE', hue='CDPHYSSEXE', width=.6, palette="vlag", showfliers=False, ax=axs[0, 0])
g.set_xlabel('Sex')
g.set_ylabel('Annual CM claims amount (MHI) (CHF)')
# Second subplot
# g = sns.boxplot(data=df_open, y="cds", x='CDPHYSSEXE',hue='CDPHYSSEXE', width=.6, palette="vlag", showfliers=False, ax=axs[0, 1])
# g.set_xlabel('Sex')
# g.set_ylabel('Chronic Disease Score (CHF)')
# Third subplot
g = sns.barplot(data=df_open, y="PRESTATIONS_BRUTES_CAM", x='CDPHYSSEXE', hue='CDPHYSSEXE', width=.6, palette="vlag", ax=axs[1, 0])
g.set_xlabel('Sex')
g.set_ylabel('Annual CAM claims amount (MHI) (CHF)')
# Fourth subplot
g = sns.boxplot(data=df_open, y="PRESTATIONS_BRUTES_LCA", x='CDPHYSSEXE', hue='CDPHYSSEXE', width=.6, palette="vlag",showfliers=False, ax=axs[1, 1])
g.set_xlabel('Sex')
g.set_ylabel('Annual CAM claims amount (SI) (CHF)')
# Adjust layout
plt.tight_layout()
plt.savefig(output_folder/'Fourplot_cost_by_sex.png', dpi = 300, bbox_inches='tight')

### Age -> cost, disease state, treatment

In [ ]:
# Create a 2x2 grid of subplots
fig, axs = plt.subplots(2, 2, figsize=(10, 6))

# First subplot
g = sns.boxplot(data=df_open, y="PRESTATIONS_BRUTES_AOS", x='age_group', width=.6, palette="vlag", showfliers=False, ax=axs[0, 0])
g.set_xlabel('Age group')
g.set_ylabel('Annual CM claims amount (MHI) (CHF)')
# Second subplot
# g = sns.boxplot(data=df_open, y="cds", x='age_group', hue='age_group',width=.6, palette="vlag", showfliers=False, ax=axs[0, 1])
# g.set_xlabel('Age group')
# g.set_ylabel('Chronic Disease Score (CHF)')
# Third subplot
g = sns.barplot(data=df_open, y="PRESTATIONS_BRUTES_CAM", x='age_group', width=.6, palette="vlag", ax=axs[1, 0])
g.set_xlabel('Age group')
g.set_ylabel('Annual CAM claims amount (MHI) (CHF)')
# Fourth subplot
g = sns.boxplot(data=df_open, y="PRESTATIONS_BRUTES_LCA", x='age_group', width=.6, palette="vlag",showfliers=False, ax=axs[1, 1])
g.set_xlabel('Age group')
g.set_ylabel('Annual CAM claims amount (SI) (CHF)')
# Adjust layout
plt.tight_layout()
plt.savefig(output_folder/'Fourplot_cost_by_age.png', dpi = 300, bbox_inches='tight')

## Time -> Cost, Disease, Treament

In [ ]:
palette = sns.color_palette()
dic_labels = {1:'Usage',0:'No usage'}

In [ ]:
# Create a 2x2 grid of subplots
fig, axs = plt.subplots(2, 2, figsize=(9, 6))

# List of dataframes and titles
dataframes = [df_open, df_healthy, df_multimorbid, df_cancer]
titles = ['All individuals', 'Individuals without PCG flag', 'Multimorbid individuals', 'Individuals with cancer']

# Create subplots
for i, (ax, df, title) in enumerate(zip(axs.flat, dataframes, titles)):
    sns.lineplot(x='NOANNEE', y='PRESTATIONS_BRUTES_AOS', hue='treatment_cam_only', data=df, marker='o', ax=ax, legend=False)
    ax.set_title(title)
    ax.set_xlabel('Year')
    ax.set_ylabel('Conventional Medicine\nexpenditures (CHF)')
    ax.grid(True)

# Manually create legend handles and labels
legend_handles = [plt.Line2D([0], [0], marker='o', color=palette[i], 
                             label=dic_labels[treatment], markersize=8, linestyle='-') 
                  for i, treatment in enumerate(df_open['treatment_cam_only'].unique())]

# Create a single legend with the correct title
legend = fig.legend(handles=legend_handles, 
           title='Complementary & Alternative Medicine (CAM)\nthrough Mandatory Health Insurance (MHI)', 
           loc='lower center', alignment='center', bbox_to_anchor=(0.5, -0.04), ncol=2)
plt.setp(legend.get_title(), ha='center')  # 'ha' stands for horizontal alignment

# Adjust layout
plt.tight_layout()
fig.subplots_adjust(bottom=0.15)  # Make room for the legend
fig.text(0.05, 0.02, 'A', fontsize=16, fontweight='bold', 
         transform=fig.transFigure, ha='left', va='bottom')

plt.savefig(model_folder/'trends_treatment_cam_only_aos_cost.png', dpi=300, bbox_inches='tight')
plt.savefig('../results/Figure 5A.png', dpi=300, bbox_inches='tight')

plt.show()

In [ ]:
# Create a 2x2 grid of subplots
fig, axs = plt.subplots(2, 2, figsize=(9, 6))

# List of dataframes and titles
dataframes = [df_5years, df_healthy_5years, df_multimorbid_5years, df_cancer_5years]
titles = ['All individuals', 'Individuals without PCG flag', 'Multimorbid individuals', 'Individuals with cancer']

# Create subplots
for i, (ax, df, title) in enumerate(zip(axs.flat, dataframes, titles)):
    sns.lineplot(x='NOANNEE', y='PRESTATIONS_BRUTES_AOS', hue='treatment_cam_only', data=df, marker='o', ax=ax, legend=False)
    ax.set_title(title)
    ax.set_xlabel('Year')
    ax.set_ylabel('Conventional Medicine\nexpenditures (CHF)')
    ax.grid(True)

# Manually create legend handles and labels
legend_handles = [plt.Line2D([0], [0], marker='o', color=palette[i], 
                             label=dic_labels[treatment], markersize=8, linestyle='-') 
                  for i, treatment in enumerate(df_open['treatment_cam_only'].unique())]

# Create a single legend with the correct title
legend = fig.legend(handles=legend_handles, 
           title='Complementary & Alternative Medicine (CAM)\nthrough Mandatory Health Insurance (MHI)', 
           loc='lower center', alignment='center', bbox_to_anchor=(0.5, -0.04), ncol=2)
plt.setp(legend.get_title(), ha='center')  # 'ha' stands for horizontal alignment

# Adjust layout
plt.tight_layout()
fig.subplots_adjust(bottom=0.15)  # Make room for the legend
fig.text(0.05, 0.02, 'A', fontsize=16, fontweight='bold', 
         transform=fig.transFigure, ha='left', va='bottom')

plt.savefig(model_folder/'trends_treatment_cam_only_aos_cost.png', dpi=300, bbox_inches='tight')
plt.savefig('../results/Figure 5A - Closed cohort.png', dpi=300, bbox_inches='tight')

plt.show()

In [ ]:
fig, axs = plt.subplots(2, 2, figsize=(9, 6))

# List of dataframes and titles
dataframes = [df_open, df_healthy, df_multimorbid, df_cancer]
titles = ['All individuals', 'Individuals without PCG flag', 'Multimorbid individuals', 'Individuals with cancer']

# Create subplots
for i, (ax, df, title) in enumerate(zip(axs.flat, dataframes, titles)):
    sns.lineplot(x='NOANNEE', y='PRESTATIONS_BRUTES_AOS', hue='treatment', data=df, marker='o', ax=ax, legend=False)
    ax.set_title(title)
    ax.set_xlabel('Year')
    ax.set_ylabel('Conventional Medicine\nexpenditures (CHF)')
    ax.grid(True)

# Manually create legend handles and labels
legend_handles = [plt.Line2D([0], [0], marker='o', color=palette[i], 
                             label=dic_labels[treatment], markersize=8, linestyle='-') 
                  for i, treatment in enumerate(df_open['treatment'].sort_values().unique())]

# Create a single legend with the correct title
legend = fig.legend(handles=legend_handles, 
           title='Complementary & Alternative Medicine (CAM)\nthrough Supplementary Insurance (SI)', 
           loc='lower center', bbox_to_anchor=(0.5, -0.04), ncol=2)
plt.setp(legend.get_title(), ha='center')  # 'ha' stands for horizontal alignment

fig.text(0.05, 0.02, 'B', fontsize=16, fontweight='bold', 
         transform=fig.transFigure, ha='left', va='bottom')
# Adjust layout
plt.tight_layout()
fig.subplots_adjust(bottom=0.15)  # Make room for the legend

plt.savefig(model_folder/'trends_treatment_aos_cost.png', dpi=300, bbox_inches='tight')
plt.savefig('../results/Figure 5B.png', dpi=300, bbox_inches='tight')

plt.show()

In [ ]:
fig, axs = plt.subplots(2, 2, figsize=(9, 6))

# List of dataframes and titles
dataframes = [df_5years, df_healthy_5years, df_multimorbid_5years, df_cancer_5years]
titles = ['All individuals', 'Individuals without PCG flag', 'Multimorbid individuals', 'Individuals with cancer']

# Create subplots
for i, (ax, df, title) in enumerate(zip(axs.flat, dataframes, titles)):
    sns.lineplot(x='NOANNEE', y='PRESTATIONS_BRUTES_AOS', hue='treatment', data=df, marker='o', ax=ax, legend=False)
    ax.set_title(title)
    ax.set_xlabel('Year')
    ax.set_ylabel('Conventional Medicine\nexpenditures (CHF)')
    ax.grid(True)

# Manually create legend handles and labels
legend_handles = [plt.Line2D([0], [0], marker='o', color=palette[i], 
                             label=dic_labels[treatment], markersize=8, linestyle='-') 
                  for i, treatment in enumerate(df_open['treatment'].sort_values().unique())]

# Create a single legend with the correct title
legend = fig.legend(handles=legend_handles, 
           title='Complementary & Alternative Medicine (CAM)\nthrough Supplementary Insurance (SI)', 
           loc='lower center', bbox_to_anchor=(0.5, -0.04), ncol=2)
plt.setp(legend.get_title(), ha='center')  # 'ha' stands for horizontal alignment

fig.text(0.05, 0.02, 'B', fontsize=16, fontweight='bold', 
         transform=fig.transFigure, ha='left', va='bottom')
# Adjust layout
plt.tight_layout()
fig.subplots_adjust(bottom=0.15)  # Make room for the legend

plt.savefig(model_folder/'trends_treatment_aos_cost.png', dpi=300, bbox_inches='tight')
plt.savefig('../results/Figure 5B - Closed cohort.png', dpi=300, bbox_inches='tight')

plt.show()

In [ ]:
fig, axs = plt.subplots(2, 2, figsize=(9, 6))
# List of dataframes and titles
dataframes = [df_open, df_healthy, df_multimorbid, df_cancer]
titles = ['All', 'Individuals without PCG flag', 'Multimorbid individuals', 'Individuals with cancer']

# Create subplots
for i, (ax, df, title) in enumerate(zip(axs.flat, dataframes, titles)):
    sns.lineplot(x='NOANNEE', y='n_atc', hue='treatment', data=df, marker='o', ax=ax, legend=False)
    ax.set_title(title)
    ax.set_xlabel('Year')
    ax.set_ylabel('Number of ATCs')
    ax.grid(True)
    
    # Force yearly ticks
    ax.set_xticks(df['NOANNEE'].unique())
    ax.set_xticklabels(df['NOANNEE'].unique())
    
# Manually create legend handles and labels
legend_handles = [plt.Line2D([0], [0], marker='o', color=palette[i], 
                             label=dic_labels[treatment], markersize=8, linestyle='-') 
                  for i, treatment in enumerate(df['treatment'].unique())]

# Create a single legend with the correct title
fig.legend(handles=legend_handles, 
           title='Complementary & Alternative Medicine (CAM)\nthrough Supplementary Insurance (SI)', 
           loc='lower center', bbox_to_anchor=(0.5, -0.04), ncol=2)
# Adjust layout
plt.tight_layout()
fig.subplots_adjust(bottom=0.15) 

fig.text(0.05, 0.02, 'B', fontsize=16, fontweight='bold', 
         transform=fig.transFigure, ha='left', va='bottom')

plt.savefig(model_folder/'trends_treatment_atcs.png', dpi=300, bbox_inches='tight')
plt.savefig(results_folder/'Figure S4B.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
fig, axs = plt.subplots(2, 2, figsize=(9, 6))
# List of dataframes and titles
dataframes = [df_open, df_healthy, df_multimorbid, df_cancer]
titles = ['All', 'Individuals without PCG flag', 'Multimorbid individuals', 'Individuals with cancer']

# Create subplots
for i, (ax, df, title) in enumerate(zip(axs.flat, dataframes, titles)):
    sns.lineplot(x='NOANNEE', y='n_atc', hue='treatment_cam_only', data=df, marker='o', ax=ax, legend=False)
    ax.set_title(title)
    ax.set_xlabel('Year')
    ax.set_ylabel('Number of ATCs')
    ax.grid(True)
    
    # Force yearly ticks
    ax.set_xticks(df['NOANNEE'].unique())
    ax.set_xticklabels(df['NOANNEE'].unique())
    
# Manually create legend handles and labels
legend_handles = [plt.Line2D([0], [0], marker='o', color=palette[i], 
                             label=dic_labels[treatment], markersize=8, linestyle='-') 
                  for i, treatment in enumerate(df['treatment_cam_only'].unique())]

# Create a single legend with the correct title
fig.legend(handles=legend_handles, 
           title='Complementary & Alternative Medicine (CAM)\nthrough Supplementary Insurance (MHI)', 
           loc='lower center', bbox_to_anchor=(0.5, -0.04), ncol=2)
# Adjust layout
plt.tight_layout()
fig.subplots_adjust(bottom=0.15)  # Make room for the legend
fig.text(0.05, 0.02, 'A', fontsize=16, fontweight='bold', 
         transform=fig.transFigure, ha='left', va='bottom')
plt.savefig(model_folder/'trends_treatment_atcs_MHI.png', dpi=300, bbox_inches='tight')
plt.savefig(results_folder/'Figure S4A.png', dpi=300, bbox_inches='tight')

plt.show()

## Cancer patients

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(15, 4))

g = sns.barplot(data=df_open, y="treatment", x='Cancer_PCG', hue='Cancer_PCG', width=.6, palette="vlag", ax=axs[0])
g.set_ylabel('Prevalence of CAM use (SI)')

g = sns.barplot(data=df_open, y="treatment_cam_only", x='Cancer_PCG', hue='Cancer_PCG', width=.6, palette="vlag", ax=axs[1])
g.set_ylabel('Prevalence of CAM use (MHI)')

g = sns.barplot(data=df_open, y="treatment_lca_cam", x='Cancer_PCG', hue='Cancer_PCG', width=.6, palette="vlag", ax=axs[2])
g.set_ylabel('Prevalence of CAM use (SI+MHI)')

plt.savefig(output_folder/'Prevalence_cancer_patients.png', dpi = 300, bbox_inches='tight')

In [ ]:
# Melt the dataframe to long format for methods columns.
methods_regex = 'Methodes.*_amount$'
methods_long_df = df_open.melt(id_vars='Cancer_PCG', 
                          value_vars=df_open.filter(regex=methods_regex).columns, 
                          var_name='Method', value_name='Amount')

In [ ]:
methods_long_df['Method'] = methods_long_df['Method'].str.replace('_amount','')

In [ ]:
# Now use Seaborn to create a bar plot with error bars.
plt.figure(figsize=(14, 7))
sns.barplot(data=methods_long_df, x='Method', y='Amount', hue='Cancer_PCG', errorbar='ci')

# Improve the aesthetics and readability.
# plt.title('Comparison of Method Amounts by Cancer Classification with Error Bars')
plt.xlabel('Methods')
plt.ylabel('Average yearly CAM claims amount (SI) (CHF)')
plt.xticks(rotation=45, ha='right')  # Rotate x-axis labels for better readability.
# plt.legend(title='Cancer_PCG')
plt.tight_layout()  # Adjust layout to prevent clipping of tick-labels.
plt.savefig(output_folder/'Costs_by_lca_method_cancer_patients.png', dpi = 300, bbox_inches='tight')

In [ ]:
for col in df_open.filter(regex=methods_regex).columns:
    df_open[f'{col}_binary'] = df_open[col].apply(lambda x: 1 if x > 0 else 0)

In [ ]:
methods_regex = 'Methodes.*_binary$'
methods_long_df = df_open.melt(id_vars='Pain_PCG', 
                          value_vars=df_open.filter(regex=methods_regex).columns, 
                          var_name='Method', value_name='Prevalence')
methods_long_df['Method'] = methods_long_df['Method'].str.replace('amount_','')

In [ ]:
methods_regex = 'Methodes.*_binary$'
methods_long_df = df_open.melt(id_vars='Cancer_PCG', 
                          value_vars=df_open.filter(regex=methods_regex).columns, 
                          var_name='Method', value_name='Prevalence')
methods_long_df['Method'] = methods_long_df['Method'].str.replace('_amount_binary','')

In [ ]:
# Now use Seaborn to create a bar plot with error bars.
plt.figure(figsize=(14, 7))
sns.barplot(data=methods_long_df, x='Method', y='Prevalence', hue='Cancer_PCG', errorbar='ci')

# Improve the aesthetics and readability.
# plt.title('Comparison of Method Amounts by Cancer Classification with Error Bars')
plt.xlabel('Methods')
plt.ylabel('Prevalence')
plt.xticks(rotation=45, ha='right')  # Rotate x-axis labels for better readability.
# plt.legend(title='Cancer_PCG')
plt.tight_layout()  # Adjust layout to prevent clipping of tick-labels.
plt.savefig(output_folder/'Prevalence_by_lca_method_cancer_patients.png', dpi = 300, bbox_inches='tight')

### Prevalence utilisation des CAM par PCG

In [ ]:
methods_regex = 'PCG'
methods_long_df = df_open.melt(id_vars='treatment_cam_only', 
                          value_vars=df_open.filter(regex=methods_regex).columns, 
                          var_name='Disease', value_name='Prevalence')
methods_long_df['Disease'] = methods_long_df['Disease'].str.replace('_PCG','')

methods_long_df = methods_long_df.rename(columns = {"treatment_cam_only":'Treatment 3 (MHI only)'})

plt.figure(figsize=(14, 7))
sns.barplot(data=methods_long_df, x='Disease', y='Prevalence', hue='Treatment 3 (MHI only)', errorbar='ci')

# Improve the aesthetics and readability.
# plt.title('Comparison of Method Amounts by Cancer Classification with Error Bars')
plt.xlabel('Methods')
plt.ylabel('Prevalence')
plt.xticks(rotation=45, ha='right')  # Rotate x-axis labels for better readability.
# plt.legend(title='Cancer_PCG')
plt.tight_layout()  # Adjust layout to prevent clipping of tick-labels.
plt.savefig(output_folder/'Prevalence_cam_only_by_pcg.png', dpi = 300, bbox_inches='tight')

In [ ]:
methods_regex = 'PCG'
methods_long_df = df_open.melt(id_vars='treatment', 
                          value_vars=df_open.filter(regex=methods_regex).columns, 
                          var_name='Disease', value_name='Prevalence')
methods_long_df['Disease'] = methods_long_df['Disease'].str.replace('_PCG','')

methods_long_df = methods_long_df.rename(columns = {"treatment":'Treatment 1 (SI only)'})

plt.figure(figsize=(14, 7))
sns.barplot(data=methods_long_df, x='Disease', y='Prevalence', hue='Treatment 1 (SI only)', errorbar='ci')

# Improve the aesthetics and readability.
# plt.title('Comparison of Method Amounts by Cancer Classification with Error Bars')
plt.xlabel('Methods')
plt.ylabel('Prevalence')
plt.xticks(rotation=45, ha='right')  # Rotate x-axis labels for better readability.
# plt.legend(title='Cancer_PCG')
plt.tight_layout()  # Adjust layout to prevent clipping of tick-labels.
plt.savefig(output_folder/'Prevalence_lca_only_by_pcg.png', dpi = 300, bbox_inches='tight')

## Heatmaps + HC - PCG vs methods (Figure 4)
### Heatmap SI

In [ ]:
methods_regex = 'Methodes.*_binary$'
# methods_long_df = df_treated_filtered.melt(id_vars='Cancer_PCG', 
#                           value_vars=df_treated_filtered.filter(regex=methods_regex).columns, 
#                           var_name='Method', value_name='Prevalence')
df_open.filter(regex=methods_regex).columns

In [ ]:
methods_regex = 'PCG'
methods_long_df = df_open.melt(id_vars=['uuid','year'], 
                          value_vars=df_open.filter(regex=methods_regex).columns, 
                          var_name='Disease category', value_name='Prevalence')
methods_long_df['Disease category'] = methods_long_df['Disease category'].str.replace('_PCG','')

In [ ]:
methods_long_df = methods_long_df[methods_long_df.Prevalence == 1]

df_test = df_open.filter(regex='Methodes.*binary$|uuid|year')

df_disease_methods = pd.merge(methods_long_df,df_test, on = ['uuid','year'])

In [ ]:
df_disease_methods[df_disease_methods['Disease category'] == 'Asthma']['Methodes ayurvediques_amount_binary'].sum()

In [ ]:
df_disease_methods[df_disease_methods['Disease category'] == 'Cancer']['Methodes ayurvediques_amount_binary'].sum()

In [ ]:
df_disease_methods = df_disease_methods.groupby('Disease category').sum()

In [ ]:
exclude_columns = ['year','uuid','Prevalence']

for col in df_disease_methods.columns:
    if col not in exclude_columns:
        print(col)
        df_disease_methods[col] = df_disease_methods[col] / df_disease_methods['Prevalence']

In [ ]:
df_disease_methods = df_disease_methods.drop(['year','Prevalence'], axis = 1)

df_disease_methods.columns = df_disease_methods.columns.str.replace('_amount_binary','')

In [ ]:
method_mapping = {
    'Methodes ayurvediques': 'Ayurvedic methods',
    "Methodes d'art therapie": 'Art therapy methods',
    'Methodes de massage': 'Massage methods',
    'Methodes energetiques': 'Energetic methods',
    'Methodes energetiques manuelles': 'Manual energetic methods',
    'Methodes hydrotherapeutiques': 'Hydrotherapeutic methods',
    'Methodes occidentales': 'Western methods',
    'Methodes orientales': 'Eastern methods',
    'Methodes prescriptives': 'Prescriptive methods',
    'Methodes psychologiques complementaires': 'Complementary psychological methods',
    'Methodes reflexes': 'Reflex methods',
    'Methodes therapeutiques par le mouvement': 'Movement-based therapeutic methods'
}

disease_mapping = {

    'Asthma' : "Asthma",
  'Diabetes' : "Diabetes",
  'Cancer'   : "Cancer",
  'Epilepsy' : "Epilepsy",
  'Glaucoma' : "Glaucoma",
  'HIV_AIDS' : "HIV/AIDS",
  'Heart_disease' : "Heart disease",
  'Hypertension_related' : "Hypertension",
  'Immune' : "Immune disorders",
  'Inflammatory' : "Inflammatory disorders",
  'Mental' : "Mental Health conditions",
  'Other' : "Other conditions",
  'Pain' : "Pain Related conditions",
  'Parkinson' : "Parkinson's disease",
  'Thyroid' : "Thyroid disorders",
}

In [ ]:
df_disease_methods = df_disease_methods.rename(columns=method_mapping)
df_disease_methods = df_disease_methods.rename(index=disease_mapping)

In [ ]:
# Better color bar handling
cmap = sns.diverging_palette(220, 20, as_cmap=True)  # Create a custom diverging palette

# Generate the clustermap
g = sns.clustermap(
    df_disease_methods.drop('uuid',axis=1),
    z_score=1,
    cmap=cmap,
    figsize=(10, 10),  # Adjust size to fit labels if needed
    linewidths=.5,  # Add lines between cells to enhance readability
    cbar_kws={'label': 'Standardized scores'}  # Label for the color bar
)
plt.setp(g.ax_heatmap.get_yticklabels(), rotation=0)  # Rotate the y labels for better readability

g.fig.text(0.1, 0.15, 'B', fontsize=16, fontweight='bold', 
         transform=fig.transFigure, ha='left', va='bottom')

# plt.savefig(results_folder/'Heatmap_disease_SI_methods.png', dpi = 300, bbox_inches='tight')
plt.savefig(results_folder/'Figure 4B.png', dpi = 300, bbox_inches='tight')

### Heatmap MHI

In [ ]:
df_cdposition = pd.read_parquet(data_folder/'processed/Intermediate datasets/df_amount_by_cdposition.parquet.gzip')

In [ ]:
columns_to_update = ['Acupuncture', 'Anthroposophic medicine', 'Homeopathy', 'Neural therapy', 'Phytotherapy', 'Traditional Chinese medicine']
df_open[[i+'_b' for i in columns_to_update]] = df_open[columns_to_update].apply(lambda x: np.where(x > 0, 1, x))

In [ ]:
methods_regex = 'Acupuncture_b|Anthroposophic medicine_b|Homeopathy_b|Neural therapy_b|Phytotherapy_b|Traditional Chinese medicine_b'
# methods_long_df = df_treated_filtered.melt(id_vars='Cancer_PCG', 
#                           value_vars=df_treated_filtered.filter(regex=methods_regex).columns, 
#                           var_name='Method', value_name='Prevalence')
df_open.filter(regex=methods_regex).columns

In [ ]:
methods_regex = 'PCG'
methods_long_df = df_open.melt(id_vars=['uuid','year'], 
                          value_vars=df_open.filter(regex=methods_regex).columns, 
                          var_name='Disease category', value_name='Prevalence')
methods_long_df['Disease category'] = methods_long_df['Disease category'].str.replace('_PCG','')

In [ ]:
methods_long_df = methods_long_df[methods_long_df.Prevalence == 1]

df_test = df_open.filter(regex='uuid|year|Acupuncture_b|Anthroposophic medicine_b|Homeopathy_b|Neural therapy_b|Phytotherapy_b|Traditional Chinese medicine_b')
# g = sns.clustermap(df_test, method="average", cmap="coolwarm", standard_scale=1)

df_disease_methods = pd.merge(methods_long_df,df_test, on = ['uuid','year'])

In [ ]:
df_disease_methods = df_disease_methods.groupby('Disease category').sum()

In [ ]:
exclude_columns = ['uuid','year','Prevalence']

for col in df_disease_methods.columns:
    if col not in exclude_columns:
        df_disease_methods[col] = df_disease_methods[col] / df_disease_methods['Prevalence']

In [ ]:
df_disease_methods = df_disease_methods.drop(['year','uuid','Prevalence'], axis = 1)
df_disease_methods.columns = df_disease_methods.columns.str.replace('_b','')

In [ ]:
# df_disease_methods = df_disease_methods.rename(columns=method_mapping)
df_disease_methods = df_disease_methods.rename(index=disease_mapping)

In [ ]:
# Generate the clustermap
cmap = sns.diverging_palette(220, 20, as_cmap=True)  # Create a custom diverging palette

g = sns.clustermap(
    df_disease_methods,
    z_score=1,
    cmap=cmap,
    figsize=(10, 10),  # Adjust size to fit labels if needed
    linewidths=.5,  # Add lines between cells to enhance readability
    cbar_kws={'label': 'Standardized scores'}  # Label for the color bar
)
plt.setp(g.ax_heatmap.get_yticklabels(), rotation=0) 

g.fig.text(0.1, 0.15, 'A', fontsize=16, fontweight='bold', 
         transform=fig.transFigure, ha='left', va='bottom')

# plt.savefig(results_folder/'Heatmap_disease_MHI_methods.png', dpi = 300, bbox_inches='tight')
plt.savefig(results_folder/'Figure 4A.png', dpi = 300, bbox_inches='tight')

## Investigate decrease in spending for cancer patients 2020-2021

In [ ]:
df_prestation_aos = read_data(data_folder/'processed'/'df_prestation_aos_preprocessed.parquet.gzip')

In [ ]:
df_prestation_aos_cancer = df_prestation_aos[df_prestation_aos.uuid.isin(df_cancer.uuid)]

In [ ]:
df_prestation_aos_cancer.groupby('treatment_Q').PRESTATIONS_BRUTES.mean()

In [ ]:
most_expensive_cat = df_prestation_aos_cancer.groupby('SOUS_CATEGORIE_DISPENSATEUR').PRESTATIONS_BRUTES.sum().sort_values(ascending=False).head(10).index

In [ ]:
df_prestation_aos_cancer[df_prestation_aos_cancer.SOUS_CATEGORIE_DISPENSATEUR.isin(most_expensive_cat)].SOUS_CATEGORIE_DISPENSATEUR.nunique()

In [ ]:
df_prestation_aos_cancer[df_prestation_aos_cancer.SOUS_CATEGORIE_DISPENSATEUR.isin(most_expensive_cat)].groupby(['ANNEE_TRAITEMENT','SOUS_CATEGORIE_DISPENSATEUR'], observed=True).PRESTATIONS_BRUTES.mean().plot.bar()

In [ ]:
_df_prestation_aos_cancer_top10 = df_prestation_aos_cancer[df_prestation_aos_cancer.SOUS_CATEGORIE_DISPENSATEUR.isin(most_expensive_cat)]
_df_prestation_aos_cancer_top10['SOUS_CATEGORIE_DISPENSATEUR'] = _df_prestation_aos_cancer_top10['SOUS_CATEGORIE_DISPENSATEUR'].astype('string')

In [ ]:
fig, ax = plt.subplots(figsize=(18,12))
sns.lineplot(x='treatmentmonth', y='PRESTATIONS_BRUTES', data=_df_prestation_aos_cancer_top10, hue='SOUS_CATEGORIE_DISPENSATEUR', palette='tab20', marker='o', ax=ax, legend=True)
plt.xticks(rotation=45, ha='right')  # Rotate x-axis labels for better readability.

In [ ]:
df_amount_by_souscat_disp = pd.read_parquet(data_folder/'processed'/'Intermediate datasets'/'df_amount_by_souscat_disp.parquet.gzip')

In [ ]:
df_cancer_providers = df_amount_by_souscat_disp[df_amount_by_souscat_disp['uuid'].isin(df_cancer['uuid'])]


In [ ]:
top_20_ss_cat_cancer = df_amount_by_souscat_disp[df_amount_by_souscat_disp.uuid.isin(df_cancer.uuid)].drop(['uuid','ANNEE_TRAITEMENT'], axis=1).sum().sort_values().tail(20).index

In [ ]:
id_vars = ['uuid', 'ANNEE_TRAITEMENT']

In [ ]:
# Create long format dataset
df_long = df_cancer_providers.melt(
    id_vars=id_vars,
    value_vars=top_20_ss_cat_cancer,
    var_name='Provider_Type',
    value_name='Amount'
)

In [ ]:
# Step 2: Calculate yearly summaries by provider type
yearly_summary = df_long.groupby(['ANNEE_TRAITEMENT', 'Provider_Type']).agg({
    'Amount': ['sum', 'mean', 'count']
}).round(2)


In [ ]:
# Flatten column names
yearly_summary.columns = ['Total_Amount', 'Mean_Amount', 'N_Claims']
yearly_summary = yearly_summary.reset_index()

In [ ]:
# Step 3: Pivot to show years as columns
pivot_total = yearly_summary.pivot(index='Provider_Type', columns='ANNEE_TRAITEMENT', values='Total_Amount')
pivot_mean = yearly_summary.pivot(index='Provider_Type', columns='ANNEE_TRAITEMENT', values='Mean_Amount')
pivot_count = yearly_summary.pivot(index='Provider_Type', columns='ANNEE_TRAITEMENT', values='N_Claims')

In [ ]:
pivot_count

In [ ]:
# Step 4: Calculate year-over-year percentage changes
def calculate_pct_change(df, years=[2019, 2020, 2021]):
    """Calculate percentage changes for specified years"""
    pct_changes = pd.DataFrame(index=df.index)
    
    for i in range(1, len(years)):
        prev_year = years[i-1]
        curr_year = years[i]
        col_name = f'Change_{prev_year}_to_{curr_year}'
        
        if prev_year in df.columns and curr_year in df.columns:
            pct_changes[col_name] = ((df[curr_year] - df[prev_year]) / df[prev_year] * 100).round(1)
    
    return pct_changes

# Calculate percentage changes for total amounts
pct_changes_total = calculate_pct_change(pivot_total)

# Step 5: Create comprehensive analysis table
def create_disruption_analysis():
    """Create a comprehensive table showing potential service disruptions"""
    
    # Combine all metrics
    analysis_df = pd.DataFrame(index=pivot_total.index)
    
    # Add yearly totals (in thousands CHF for readability)
    for year in [2017, 2018, 2019, 2020, 2021]:
        if year in pivot_total.columns:
            analysis_df[f'Total_{year}_k'] = (pivot_total[year] / 1000).round(1)
    
    # Add year-over-year changes
    analysis_df = pd.concat([analysis_df, pct_changes_total], axis=1)
    
    # Add pre-pandemic vs pandemic comparison (2019 vs 2020)
    if 2019 in pivot_total.columns and 2020 in pivot_total.columns:
        analysis_df['Pandemic_Impact_2020'] = pct_changes_total['Change_2019_to_2020']
    
    # Add recovery analysis (2020 vs 2021)
    if 2020 in pivot_total.columns and 2021 in pivot_total.columns:
        analysis_df['Recovery_2021'] = pct_changes_total['Change_2020_to_2021']
    
    # Sort by 2019 total spending (pre-pandemic baseline)
    if 'Total_2019_k' in analysis_df.columns:
        analysis_df = analysis_df.sort_values('Total_2019_k', ascending=False)
    
    return analysis_df

# Create the main analysis table
disruption_table = create_disruption_analysis()

# Step 6: Identify major disruptions
def identify_disruptions(df, threshold_decline=-15):
    """Identify provider types with significant disruptions"""
    
    disrupted_services = []
    
    for provider in df.index:
        # Check for 2021 decline
        if 'Recovery_2021' in df.columns:
            change_2021 = df.loc[provider, 'Recovery_2021']
            if pd.notna(change_2021) and change_2021 < threshold_decline:
                disrupted_services.append({
                    'Provider': provider,
                    'Change_2021': change_2021,
                    'Type': 'Decline in 2021'
                })
        
        # Check for major pandemic impact
        if 'Pandemic_Impact_2020' in df.columns:
            pandemic_impact = df.loc[provider, 'Pandemic_Impact_2020']
            if pd.notna(pandemic_impact) and pandemic_impact < threshold_decline:
                disrupted_services.append({
                    'Provider': provider,
                    'Change_2020': pandemic_impact,
                    'Type': 'Pandemic disruption 2020'
                })
    
    return pd.DataFrame(disrupted_services)

# Identify disrupted services
disrupted_services = identify_disruptions(disruption_table)

# Step 7: Display results
print("=== CANCER CARE PROVIDER TRENDS ANALYSIS ===\n")

print("1. COMPREHENSIVE TRENDS TABLE")
print("(Amounts in thousands CHF)")
print(disruption_table.to_string())

print("\n\n2. SERVICES WITH SIGNIFICANT DISRUPTIONS")
if not disrupted_services.empty:
    print(disrupted_services.to_string(index=False))
else:
    print("No major disruptions identified with current threshold")

print("\n\n3. SUMMARY STATISTICS")
print(f"Total provider types analyzed: {len(disruption_table)}")

if 'Recovery_2021' in disruption_table.columns:
    decline_2021 = (disruption_table['Recovery_2021'] < 0).sum()
    print(f"Provider types with declining spending in 2021: {decline_2021}")
    
    avg_change_2021 = disruption_table['Recovery_2021'].mean()
    print(f"Average change in 2021: {avg_change_2021:.1f}%")

# Step 8: Export for manuscript
disruption_table.to_csv('cancer_provider_trends_analysis.csv')
print(f"\nTable exported to: cancer_provider_trends_analysis.csv")

# Optional: Create a focused table for specific providers of interest
key_oncology_providers = [col for col in top_20_ss_cat_cancer 
                         if any(keyword in col.lower() for keyword in 
                               ['onco', 'cancer', 'chimioth', 'radioth', 'hémato'])]

if key_oncology_providers:
    print(f"\n\n4. FOCUSED ONCOLOGY SERVICES ANALYSIS")
    print("Key oncology providers identified:")
    oncology_focused = disruption_table.loc[key_oncology_providers]
    print(oncology_focused.to_string())

In [ ]:
# Analyze pandemic impact across different patient subgroups
# This will prove whether cancer services were uniquely vulnerable

# Step 1: Define patient subgroups and their datasets
subgroups = {
    'All individuals': df_open,
    'Individuals without PCG flag': df_healthy, 
    'Multimorbid individuals': df_multimorbid,
    'Individuals with cancer': df_cancer
}

# Step 2: First, identify top 20 providers for each subgroup
def get_top_providers_for_subgroup(subgroup_data, n_top=20):
    """Get top N provider types by total expenditure for a specific subgroup"""
    
    # Filter to this subgroup
    df_subgroup = df_amount_by_souscat_disp[
        df_amount_by_souscat_disp['uuid'].isin(subgroup_data['uuid'])
    ]
    
    # Get all provider columns (exclude uuid, year columns)
    provider_columns = [col for col in df_subgroup.columns 
                       if col not in ['uuid', 'ANNEE_TRAITEMENT']]
    
    # Calculate total expenditure by provider type
    provider_totals = df_subgroup[provider_columns].sum().sort_values(ascending=False)
    
    # Return top N providers
    return provider_totals.head(n_top).index.tolist()

# Get top 20 providers for each subgroup
subgroup_top_providers = {}
for subgroup_name, subgroup_data in subgroups.items():
    top_providers = get_top_providers_for_subgroup(subgroup_data)
    subgroup_top_providers[subgroup_name] = top_providers
    print(f"\nTop 10 providers for {subgroup_name}:")
    for i, provider in enumerate(top_providers[:10], 1):
        print(f"  {i}. {provider}")

# Step 3: Calculate 2021 changes for each subgroup using their own top providers
def calculate_pandemic_impact_by_subgroup(subgroup_data, subgroup_name, top_providers):
    """Calculate 2021 changes for a specific patient subgroup"""
    
    # Filter to this subgroup
    df_subgroup = df_amount_by_souscat_disp[
        df_amount_by_souscat_disp['uuid'].isin(subgroup_data['uuid'])
    ]
    
    # Melt to long format
    df_long = df_subgroup.melt(
        id_vars=['uuid', 'ANNEE_TRAITEMENT'],
        value_vars=top_providers,
        var_name='Provider_Type',
        value_name='Amount'
    )
    
    # Calculate yearly totals by provider
    yearly_totals = df_long.groupby(['ANNEE_TRAITEMENT', 'Provider_Type'])['Amount'].sum().reset_index()
    
    # Pivot to get years as columns
    pivot_data = yearly_totals.pivot(index='Provider_Type', columns='ANNEE_TRAITEMENT', values='Amount')
    
    # Calculate 2020->2021 change
    if 2020 in pivot_data.columns and 2021 in pivot_data.columns:
        change_2021 = ((pivot_data[2021] - pivot_data[2020]) / pivot_data[2020] * 100).round(1)
    else:
        change_2021 = pd.Series(dtype=float)
    
    return change_2021.to_frame(f'{subgroup_name}_2021_change')

# Step 4: Calculate changes for each subgroup using their own top providers
subgroup_changes = {}
for subgroup_name, subgroup_data in subgroups.items():
    top_providers = subgroup_top_providers[subgroup_name]
    changes = calculate_pandemic_impact_by_subgroup(subgroup_data, subgroup_name, top_providers)
    subgroup_changes[subgroup_name] = changes

# Step 5: For comparison, also analyze common providers across groups
# Find providers that appear in multiple subgroups' top lists
all_providers = set()
for providers in subgroup_top_providers.values():
    all_providers.update(providers)

provider_frequency = {}
for provider in all_providers:
    count = sum(1 for providers in subgroup_top_providers.values() if provider in providers)
    provider_frequency[provider] = count

# Get providers that appear in at least 2 subgroups
common_providers = [provider for provider, freq in provider_frequency.items() if freq >= 2]
print(f"\nCommon providers (appearing in ≥2 subgroups): {len(common_providers)}")

# Calculate changes for common providers across all subgroups
common_changes = []
for subgroup_name, subgroup_data in subgroups.items():
    changes = calculate_pandemic_impact_by_subgroup(subgroup_data, subgroup_name, common_providers)
    common_changes.append(changes)

pandemic_comparison_common = pd.concat(common_changes, axis=1)

# Step 6: Analyze each subgroup's specific patterns
def analyze_subgroup_specific_patterns():
    """Analyze pandemic impact using each subgroup's own top providers"""
    
    subgroup_summaries = []
    
    for subgroup_name, changes_df in subgroup_changes.items():
        change_col = changes_df.columns[0]  # The change column
        
        summary = {
            'Subgroup': subgroup_name,
            'Total_providers': len(changes_df),
            'Declining_services': (changes_df[change_col] < 0).sum(),
            'Major_declines_>15%': (changes_df[change_col] < -15).sum(),
            'Major_declines_>25%': (changes_df[change_col] < -25).sum(),
            'Median_change': changes_df[change_col].median(),
            'Worst_decline': changes_df[change_col].min(),
        }
        
        subgroup_summaries.append(summary)
    
    return pd.DataFrame(subgroup_summaries)

subgroup_summary = analyze_subgroup_specific_patterns()

# Step 7: Analyze common providers for direct comparison
def analyze_common_providers():
    """Analyze how the same providers were affected across different patient groups"""
    
    if len(common_providers) > 0 and not pandemic_comparison_common.empty:
        comparison_results = []
        
        for provider in pandemic_comparison_common.index:
            provider_data = {'Provider': provider}
            
            for col in pandemic_comparison_common.columns:
                subgroup = col.replace('_2021_change', '')
                change = pandemic_comparison_common.loc[provider, col]
                provider_data[f'{subgroup}_change'] = change
            
            comparison_results.append(provider_data)
        
        return pd.DataFrame(comparison_results)
    else:
        return pd.DataFrame()

common_provider_analysis = analyze_common_providers()

# Step 8: Display results
print("=== PANDEMIC IMPACT COMPARISON ACROSS PATIENT SUBGROUPS ===\n")

print("1. SUBGROUP-SPECIFIC ANALYSIS (each using their own top 20 providers)")
print(subgroup_summary.to_string(index=False))

print("\n\n2. TOP 5 MOST AFFECTED PROVIDERS BY SUBGROUP:")
for subgroup_name, changes_df in subgroup_changes.items():
    change_col = changes_df.columns[0]
    worst_affected = changes_df.nsmallest(5, change_col)
    print(f"\n{subgroup_name}:")
    for provider, change in worst_affected.iterrows():
        print(f"  - {provider}: {change[change_col]:.1f}%")

if not common_provider_analysis.empty:
    print(f"\n\n3. COMMON PROVIDERS COMPARISON ({len(common_providers)} providers)")
    print("(Same providers across different patient groups)")
    print(common_provider_analysis.round(1).to_string(index=False))

print(f"\n\n4. KEY FINDINGS FOR REVIEWER RESPONSE:")

# Find cancer subgroup data
cancer_summary = subgroup_summary[subgroup_summary['Subgroup'].str.contains('Cancer')]
if not cancer_summary.empty:
    cancer_row = cancer_summary.iloc[0]
    print(f"Cancer patients: {cancer_row['Declining_services']}/{cancer_row['Total_providers']} services declined")
    print(f"Cancer patients: {cancer_row['Major_declines_>15%']} services with >15% decline")
    print(f"Cancer patients: median change = {cancer_row['Median_change']:.1f}%")

print(f"\nOther subgroups comparison:")
for _, row in subgroup_summary.iterrows():
    if 'Cancer' not in row['Subgroup']:
        print(f"  {row['Subgroup']}: {row['Declining_services']}/{row['Total_providers']} declining, "
              f"{row['Major_declines_>15%']} major declines, median = {row['Median_change']:.1f}%")

# Evidence for unique cancer vulnerability
print(f"\n\n5. EVIDENCE FOR UNIQUE CANCER VULNERABILITY:")
cancer_declines = cancer_summary['Declining_services'].iloc[0] if not cancer_summary.empty else 0
other_declines = subgroup_summary[~subgroup_summary['Subgroup'].str.contains('Cancer')]['Declining_services'].mean()

print(f"Cancer patients had {cancer_declines} declining services")
print(f"Other subgroups averaged {other_declines:.1f} declining services")
print(f"Cancer patients were {cancer_declines/other_declines:.1f}x more likely to experience service declines")

# Export both analyses
for subgroup_name, changes_df in subgroup_changes.items():
    filename = f'pandemic_impact_{subgroup_name.replace(" ", "_").lower()}.csv'
    changes_df.to_csv(results_folder/filename)

if not common_provider_analysis.empty:
    common_provider_analysis.to_csv('../results/pandemic_impact_common_providers.csv')

subgroup_summary.to_csv('../results/pandemic_impact_summary_by_subgroup.csv')

print(f"\nDetailed tables exported for manuscript use")